# Final D1 hand-off and arbitrary-MP4 inference

The final OOF protocol selected D1. This notebook trains the deployable all-600-video head with the frozen recipe, then exposes `predict_full_mp4(...)`. The all-data training is not a new independent metric.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from v3_final_d1_handoff import run_final_handoff
from final_d1_inference import load_final_d1, predict_full_mp4

RUN_ALL_DATA_TRAINING = True
EXTERNAL_MP4_PATH: str | None = None

In [ ]:
# 1) Train the selected D1 head on all 600 videos, using fixed epochs derived from the OOF folds.
if RUN_ALL_DATA_TRAINING:
    handoff = run_final_handoff()
    print(json.dumps(handoff, ensure_ascii=False, indent=2))
else:
    print('Set RUN_ALL_DATA_TRAINING=True to create the final model artifact.')

In [ ]:
# 2) Load the artifact once, then predict any valid MP4 without label, event time, or metadata.
predictor = load_final_d1()
if EXTERNAL_MP4_PATH is None:
    print('Set EXTERNAL_MP4_PATH to an MP4 path, then rerun this cell.')
else:
    result = predict_full_mp4(EXTERNAL_MP4_PATH, predictor)
    print({key: value for key, value in result.items() if key != 'window_predictions'})

The decision threshold is the D1 OOF-selected threshold (0.47); it was not tuned on the new MP4. The returned probability is a top-3 mean over sliding 5-second windows.